In [1]:
import requests
import tkinter as tk
from tkinter import messagebox

# API URLs
GEO_API_URL = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"

# Function to fetch location coordinates
def fetch_location_data(city_name):
    try:
        url = f"{GEO_API_URL}?name={city_name}"
        response = requests.get(url)
        data = response.json()

        if "results" in data and len(data["results"]) > 0:
            location = data["results"][0]
            return location["latitude"], location["longitude"]
        else:
            return None, None
    except Exception as e:
        messagebox.showerror("Error", f"Failed to fetch location data.\n{str(e)}")
        return None, None

# Function to fetch weather data
def fetch_weather_data(latitude, longitude):
    try:
        url = f"{WEATHER_API_URL}?latitude={latitude}&longitude={longitude}&current_weather=true"
        response = requests.get(url)
        data = response.json()

        if "current_weather" in data:
            weather = data["current_weather"]
            return weather["temperature"], 50.0, weather["windspeed"]  # Simulated Humidity
        else:
            return None, None, None
    except Exception as e:
        messagebox.showerror("Error", f"Failed to fetch weather data.\n{str(e)}")
        return None, None, None

# Function to fetch and display weather details
def get_weather():
    city_name = city_entry.get().strip()
    if not city_name:
        messagebox.showwarning("Input Error", "Please enter a city name.")
        return

    latitude, longitude = fetch_location_data(city_name)
    if latitude is None or longitude is None:
        messagebox.showerror("Error", "Could not retrieve location data. Please check the city name.")
        return

    temperature, humidity, wind_speed = fetch_weather_data(latitude, longitude)
    if temperature is None:
        messagebox.showerror("Error", "Could not retrieve weather data.")
        return

    result_label.config(text=f"🌡 Temperature: {temperature} °C\n💧 Humidity: {humidity} %\n💨 Wind Speed: {wind_speed} km/h",
                        fg="white")

# GUI Setup
root = tk.Tk()
root.title("Live Weather Report")
root.geometry("400x250")
root.configure(bg="#222")  # Dark background

frame = tk.Frame(root, bg="#222")
frame.pack(pady=20)

tk.Label(frame, text="🌍 Enter City Name:", font=("Arial", 12), fg="#fff", bg="#222").pack()
city_entry = tk.Entry(frame, font=("Arial", 12), width=25, bg="#333", fg="#fff", insertbackground="white")
city_entry.pack(pady=5)

tk.Button(frame, text="🔍 Get Weather", font=("Arial", 12, "bold"), bg="#008CBA", fg="white", command=get_weather).pack(pady=10)

result_label = tk.Label(root, text="Weather info will appear here", font=("Arial", 14), fg="#aaa", bg="#222")
result_label.pack(pady=10)

root.mainloop()